In [ ]:
path_to_dataset = "" #FIXME: Add path to dataset here

In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
import optuna
import sys

%load_ext autoreload
%autoreload 2

# Import the class we just created
from Legacy_pipeline import LegacyPipeline

**Helper functions**

In [ ]:
def calculate_iou(boxA, boxB):
    """Calculates Intersection over Union for two bounding boxes."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    iou = interArea / float(boxAArea + boxBArea - interArea + 1e-5)
    return iou

def yolo_to_xyxy(x_center, y_center, w, h, img_w, img_h):
    """Converts normalized YOLO [x_center, y_center, w, h] to absolute [x1, y1, x2, y2]."""
    return [
        int((x_center - w/2) * img_w),
        int((y_center - h/2) * img_h),
        int((x_center + w/2) * img_w),
        int((y_center + h/2) * img_h)
    ]

**Optuna objective function**

In [ ]:
def objective(trial, threshold_method):
    # 1. Load Data
    csv_path = "../Training_data/notch_classification_dataset.csv"
    
    df = pd.read_csv(csv_path)
    
    if 'split' in df.columns:
        df = df[df['split'] == 'train']
        
    pipeline = LegacyPipeline()
    
    # 2. Let Optuna suggest parameters   
    use_adaptive = (threshold_method == "adaptive")
    
    # Common parameters
    notch_band_width = trial.suggest_int("notch_band_width", 0, 250, step=25)
    filter_min_area = trial.suggest_int("filter_min_area", 50, 800, step=50)
    filter_max_area = trial.suggest_int("filter_max_area", 1000, 5000, step=500)
    box_padding = trial.suggest_int("box_padding", 0, 30, step=2)
    
    # Method-specific parameters
    if use_adaptive:
        # block_size must be an odd number. Optuna suggests an integer, we make it odd.
        block_size_base = trial.suggest_int("adaptive_block_size_base", 1, 45)
        adaptive_block_size = block_size_base * 2 + 1 
        adaptive_c = trial.suggest_int("adaptive_c", -5, 20)
        adaptive_use_gaussian = trial.suggest_categorical("adaptive_use_gaussian", [True, False])
        peak_safety_margin = 10 # Dummy value
    else:
        peak_safety_margin = trial.suggest_int("peak_safety_margin", 0, 30)
        adaptive_block_size = 21 # Dummy value
        adaptive_c = 10          # Dummy value
        adaptive_use_gaussian = False # Dummy value

    # Tracking metrics
    tp, fp, fn = 0, 0, 0
    iou_threshold = 0.15 # Minimum overlap to consider it a correct detection
    
    unique_images = df['image_filename'].unique()[:150]

    # 3. Evaluate the pipeline with these parameters
    for img_file in unique_images:
        subfolder = img_file[0].upper()
        img_path = os.path.join(path_to_dataset, subfolder, img_file)
        img = cv2.imread(img_path)
        
        if img is None:
            continue
            
        img_h, img_w = img.shape[:2]
        
        # Run Pipeline
        try:
            results = pipeline.process_image(
                img=img,
                use_adaptive_threshold=use_adaptive,
                notch_band_width=notch_band_width,
                adaptive_block_size=adaptive_block_size,
                adaptive_c=adaptive_c,
                adaptive_use_gaussian=adaptive_use_gaussian,
                peak_safety_margin=peak_safety_margin,
                filter_min_area=filter_min_area,
                filter_max_area=filter_max_area,
                box_padding=box_padding,
                shape_min_area=50, 
                shape_epsilon_mult=0.04
            )
        except Exception:
            # If a parameter combination crashes OpenCV penalize the trial heavily
            return 0.0

        predicted_boxes = [n['coords'] for n in results.get('accepted_notches', [])]
        
        # Get Ground Truth (only notches, ignore noise labels)
        group = df[df['image_filename'] == img_file]
        gt_notches = group[group['super_category'].str.lower() == 'notch']
        
        matched_preds = set()
        
        # Calculate TP, FP, FN
        for _, gt_row in gt_notches.iterrows():
            gt_box = yolo_to_xyxy(gt_row['x_center'], gt_row['y_center'], gt_row['width'], gt_row['height'], img_w, img_h)
            
            best_iou = 0
            best_pred_idx = -1
            
            for idx, pred_box in enumerate(predicted_boxes):
                if idx in matched_preds:
                    continue
                iou = calculate_iou(pred_box, gt_box)
                if iou > best_iou:
                    best_iou = iou
                    best_pred_idx = idx
            
            if best_iou >= iou_threshold:
                tp += 1
                matched_preds.add(best_pred_idx)
            else:
                fn += 1 # Ground truth missed
                
        # Any predicted boxes that didn't match a ground truth are false positives
        fp += len(predicted_boxes) - len(matched_preds)

    # 4. Calculate F2 Score
    denom = (5 * tp) + (4 * fn) + fp
    
    if denom == 0:
        f2_score = 0.0
    else:
        f2_score = (5 * tp) / denom
        
    return f2_score

In [ ]:
# ---------------------------------------------------------
# RUN 1: ADAPTIVE THRESHOLDING
# ---------------------------------------------------------
print("\n" + "="*50)
print("STARTING RUN 1: ADAPTIVE THRESHOLDING")
print("="*50)

study_adaptive = optuna.create_study(study_name="legacy_adaptive", direction="maximize")
# Using lambda to pass the specific method to the objective function
study_adaptive.optimize(lambda trial: objective(trial, "adaptive"), n_trials=200, n_jobs=-1, show_progress_bar=True)

# ---------------------------------------------------------
# RUN 2: PEAK DECAY THRESHOLDING
# ---------------------------------------------------------
print("\n" + "="*50)
print("STARTING RUN 2: PEAK DECAY THRESHOLDING")
print("="*50)

study_peak = optuna.create_study(study_name="legacy_peak_decay", direction="maximize")
study_peak.optimize(lambda trial: objective(trial, "peak_decay"), n_trials=200, n_jobs=-1, show_progress_bar=True)

In [ ]:
study_adaptive.trials_dataframe().to_csv("optuna_adaptive_results.csv", index=False)
study_peak.trials_dataframe().to_csv("optuna_peak_decay_results.csv", index=False)

In [ ]:
# ---------------------------------------------------------
# FINAL RESULTS
# ---------------------------------------------------------
print("\n" + "="*50)
print("FINAL COMPARISON")
print("="*50)

print(f"\nBEST ADAPTIVE F2-SCORE: {study_adaptive.best_value:.4f}")
print("Adaptive Parameters:")
for key, value in study_adaptive.best_params.items():
    if key == "adaptive_block_size_base":
        value = value * 2 + 1
    print(f"    {key}: {value}")
    
print(f"\nBEST PEAK DECAY F2-SCORE: {study_peak.best_value:.4f}")
print("Peak Decay Parameters:")
for key, value in study_peak.best_params.items():
    print(f"    {key}: {value}")